# CogniFusion — Colab Notebook

**Real-time cognitive state monitoring** fusing EEG + eye-tracking to classify
**low / medium / high** cognitive load.

| | |
|---|---|
| Repo | [PoornaSri26/CogniFusion](https://github.com/PoornaSri26/CogniFusion) |
| Runtime | CPU is fine — pick **GPU** in Runtime → Change runtime type to speed up training |
| Data | Synthetic 30-subject dataset generated in-notebook (no hardware needed) |

**Pipeline:** synthetic data → signal & feature exploration → fused EEG+eye features →
AttentionFusionNet (vector-gated cross-modal fusion, focal loss, mixup, temperature
scaling, EMA smoothing) → training curves → held-out-subject evaluation.

## 1 · Setup — clone repo & install deps

In [ ]:
# Clone the repository
!git clone -q https://github.com/PoornaSri26/CogniFusion.git
%cd CogniFusion

# Core ML deps. torch is preinstalled on Colab — don't reinstall it.
!pip install -q numpy pandas scipy scikit-learn pyarrow

import sys, torch
sys.path.insert(0, "src")
sys.path.insert(0, "data")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 2 · Generate the synthetic dataset

30 subjects × 4 epochs × 3 classes = **360 synchronized EEG+eye epochs**.
Each subject gets unique neural parameters (IAF 9–12 Hz, noise floor,
electrode offsets, pupil baseline) so the model learns subject-invariant features.

In [ ]:
from simulate_data import generate_dataset
manifest = generate_dataset("data/raw")
manifest.head()

## 3 · Build the fused feature table

EEG features (band powers, differential entropy, connectivity, entropy) + eye features (pupillometry, fixations, blinks, gaze kinematics) → one parquet table.

> This is the slowest step (~10 min on CPU) — feature extraction is per-epoch.

In [ ]:
from build_dataset import build
df = build("data/raw/manifest.csv", "data/processed/features.parquet")
df.shape

## 4 · Dataset overview

In [ ]:
import pandas as pd

df = pd.read_parquet("data/processed/features.parquet")
print("Label distribution:")
display(df["label"].value_counts().to_frame("count"))

feat_cols = [c for c in df.columns if c.startswith(("eeg__", "eye__"))]
print(f"\n{len(feat_cols)} fused features | missing values: {int(df[feat_cols].isna().sum().sum())}")

load = df["label"].map({"low": 0, "medium": 1, "high": 2})
top = df[feat_cols].corrwith(load).abs().sort_values(ascending=False).head(10)
print("\nTop features by |correlation| with cognitive load:")
display(top.to_frame("|r|"))

## 5 · Signal explorer — what does load look like in the raw signals?

One epoch per class: EEG time series + power spectral density (1–45 Hz),
then pupil diameter and gaze position.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import welch

manifest = pd.read_csv("data/raw/manifest.csv")
CHANNELS = ["AF7", "AF8", "TP9", "TP10"]

fig, axes = plt.subplots(3, 3, figsize=(17, 9))
for col, label in enumerate(["low", "medium", "high"]):
    row = manifest[manifest["label"] == label].iloc[0]
    eeg = np.load(row["eeg_path"])                      # (4, 1024)
    t = np.arange(eeg.shape[1]) / 256.0

    for ch in range(4):
        axes[0, col].plot(t, eeg[ch], lw=0.5, label=CHANNELS[ch])
    axes[0, col].set_title(f"EEG time series — {label}")
    axes[0, col].legend(fontsize=6, ncol=2)

    for ch in range(4):
        f, p = welch(eeg[ch], fs=256, nperseg=256)
        axes[1, col].semilogy(f, p, lw=0.8, label=CHANNELS[ch])
    axes[1, col].set_xlim(1, 45); axes[1, col].set_xlabel("Hz")
    axes[1, col].set_title("EEG power spectral density")
    axes[1, col].legend(fontsize=6, ncol=2)

    eye = pd.read_csv(row["eye_path"])
    axes[2, col].plot(eye["t"], eye["pupil_mm"], lw=0.6, color="steelblue", label="pupil mm")
    ax2 = axes[2, col].twinx()
    ax2.plot(eye["t"], eye["gaze_x"], lw=0.4, alpha=0.6, color="darkorange", label="gaze x")
    axes[2, col].set_title(f"Eye — pupil & gaze ({label})")
    axes[2, col].set_xlabel("s")
plt.tight_layout(); plt.show()

## 6 · Feature landscape — which features carry the load signal?

Left: top features ranked by correlation with load. Right: class means for the
top 6 — watch theta/beta rise and alpha fall as load increases.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

df = pd.read_parquet("data/processed/features.parquet")
feat_cols = [c for c in df.columns if c.startswith(("eeg__", "eye__"))]
load = df["label"].map({"low": 0, "medium": 1, "high": 2})

corr = df[feat_cols].corrwith(load)
top12 = corr.abs().sort_values(ascending=False).head(12).index

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
corr.loc[top12].sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Top 12 features — correlation with cognitive load")
axes[0].set_xlabel("Pearson r"); axes[0].axvline(0, color="k", lw=0.6)

top6 = corr.abs().sort_values(ascending=False).head(6).index
df.groupby("label")[list(top6)].mean().T.plot(kind="bar", ax=axes[1],
                                             figsize=(12, 5), colormap="viridis")
axes[1].set_title("Class means — top 6 features"); axes[1].set_ylabel("mean value")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

## 7 · Train the model

Single subject-wise holdout by default — the test subjects are **never seen in
training**, so the score measures real generalization. For a rigorous estimate,
add `--cv 5` (5-fold subject-wise CV, ~5× slower).

In [ ]:
# ~200 epochs on 360 samples — a few minutes on GPU, longer on CPU.
!python src/train.py --features data/processed/features.parquet \
                    --epochs 200 --cv 0 \
                    --out models/fusion_model

In [ ]:
import torch, json
ckpt = torch.load("models/fusion_model.pt", map_location="cpu", weights_only=False)
print(json.dumps(ckpt["cv_summary"], indent=2))

## 8 · Training curves

Loss & validation macro-F1 per eval point (every 10 epochs), recorded in the
checkpoint by `train.py`.

In [ ]:
import torch
import matplotlib.pyplot as plt

ckpt = torch.load("models/fusion_model.pt", map_location="cpu", weights_only=False)
for i, rec in enumerate(ckpt["folds"]):
    h = rec.get("history") or []
    if not h:
        print(f"Fold {i+1}: no history recorded — retrain to enable curves")
        continue
    ep   = [d["epoch"] for d in h]
    loss = [d["loss"]   for d in h]
    f1   = [d["val_f1"] for d in h]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep, loss, marker="o", ms=3, color="steelblue")
    ax[0].set_title(f"Fold {i+1} — training loss"); ax[0].set_xlabel("epoch")
    ax[1].plot(ep, f1, marker="o", ms=3, color="seagreen")
    ax[1].axhline(0.9, ls="--", c="r", alpha=0.4, label="target 0.9")
    ax[1].set_title("validation macro-F1"); ax[1].set_xlabel("epoch")
    ax[1].legend(); plt.tight_layout(); plt.show()

## 9 · Held-out-subject evaluation

Rebuilds the exact same split the model was validated on, and applies the full
training-time inference path: per-subject median/MAD normalization → per-fold
scalers → vector-gated fusion → temperature scaling → EMA smoothing.

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from model import AttentionFusionNet
from train import (subject_wise_split, adaptive_subject_norm,
                   coerce_numeric, make_norm_key, TemperatureScaler)

def load_ensemble(path="models/fusion_model.pt"):
    # Load the self-contained ensemble checkpoint (same logic as
    # realtime_pipeline.load_model, without hardware deps like brainflow).
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    folds = []
    for rec in ckpt["folds"]:
        m = AttentionFusionNet(
            eeg_dim=len(ckpt["eeg_cols"]),
            eye_dim=len(ckpt["eye_cols"]),
            embed_dim=rec.get("embed_dim", ckpt.get("embed_dim", 48)),
            n_classes=len(ckpt["label_classes"]),
            gate_type=ckpt.get("gate_type", "vector"),
        )
        m.load_state_dict(rec["model_state"])
        m.eval()
        folds.append({"model": m, "scaler_eeg": rec["scaler_eeg"],
                      "scaler_eye": rec["scaler_eye"],
                      "temp": rec.get("temp_scaler")})
    return {"folds": folds, "eeg_cols": ckpt["eeg_cols"],
            "eye_cols": ckpt["eye_cols"],
            "label_classes": list(ckpt["label_classes"])}

bundle = load_ensemble("models/fusion_model.pt")
eeg_cols, eye_cols, classes = bundle["eeg_cols"], bundle["eye_cols"], bundle["label_classes"]

# --- Same subject-wise holdout the model was validated on ---
df = pd.read_parquet("data/processed/features.parquet")
df["norm_key"] = make_norm_key(df)
_, test_df = subject_wise_split(df, test_frac=0.25, seed=42)
print(f"Held-out test set: {len(test_df)} epochs from "
      f"{test_df['subject'].nunique()} unseen subjects")

# --- Per-subject adaptive normalization (median/MAD), as in training ---
eeg_norm, _ = adaptive_subject_norm(coerce_numeric(test_df, eeg_cols), test_df["norm_key"])
eye_norm, _ = adaptive_subject_norm(coerce_numeric(test_df, eye_cols), test_df["norm_key"])

# --- Ensemble inference: mean calibrated softmax across folds ---
raw = np.zeros((len(test_df), len(classes)))
with torch.no_grad():
    for fold in bundle["folds"]:
        xe = np.nan_to_num(fold["scaler_eeg"].transform(eeg_norm))
        ye = np.nan_to_num(fold["scaler_eye"].transform(eye_norm))
        logits = fold["model"](torch.tensor(xe, dtype=torch.float32),
                               torch.tensor(ye, dtype=torch.float32))
        if fold.get("temp"):
            ts = TemperatureScaler()
            ts.load_state_dict(fold["temp"])
            ts.eval()
            logits = ts(logits)
        raw += torch.softmax(logits, dim=1).numpy()
raw /= len(bundle["folds"])

# EMA temporal smoothing (identical to training-time eval)
probs = np.zeros_like(raw)
probs[0] = raw[0]
for t in range(1, len(raw)):
    probs[t] = 0.65 * raw[t] + 0.35 * probs[t - 1]

pred = np.array(classes)[probs.argmax(1)]
acc = accuracy_score(test_df["label"], pred)
print(f"\nEnsemble accuracy on held-out subjects: {acc:.1%}")
print(classification_report(test_df["label"], pred, digits=3))

out = test_df[["subject", "label", "epoch_id"]].copy()
out["pred"] = pred
out["confidence"] = probs.max(1).round(3)
display(out.head(12))

ConfusionMatrixDisplay(
    confusion_matrix(test_df["label"], pred, labels=classes),
    display_labels=classes).plot()
plt.title("CogniFusion — held-out subject confusion matrix")
plt.show()

## 10 · Next steps

- **Real data**: `python src/build_dataset_real.py` builds the fused table from the
  1.5 GB real Muse-EEG + webcam dataset (`data/raw/dataset.csv`, 113 participants —
  see the repo README), then train with
  `--features data/processed/features_real.parquet --cv 5`.
- **Live demo**: run `python src/realtime_pipeline.py --eeg synthetic --eye synthetic`
  locally, or `streamlit run dashboard.py` for the monitoring dashboard. Webcam
  eye-tracking works in Colab (`src/webcam_eyetracker.py`); EEG needs Muse hardware
  (BrainFlow).
- **Full training**: `--epochs 300 --cv 5` for the production-grade subject-wise CV.